#### First the understatapi

-- This package is not actively maintained so use at your own risk

In [1]:
import pandas as pd
import understatapi

client = understatapi.UnderstatClient()

In [4]:
# Let's first get data from a league to retrieve a list of matches
league_data = client.league(league="EPL").get_match_data(season="2025")
league_data[0]

{'id': '26602',
 'isResult': True,
 'h': {'id': '89', 'title': 'Manchester United', 'short_title': 'MUN'},
 'a': {'id': '228', 'title': 'Fulham', 'short_title': 'FLH'},
 'goals': {'h': '1', 'a': '0'},
 'xG': {'h': '2.04268', 'a': '0.418711'},
 'datetime': '2024-08-16 19:00:00',
 'forecast': {'w': '0.8069', 'd': '0.1489', 'l': '0.0442'}}

In [7]:
# now we can use the match id to get the match data
shot_data = client.match(match="26602").get_shot_data()
shot_data['h'][0]

{'id': '584630',
 'minute': '19',
 'result': 'MissedShots',
 'X': '0.970999984741211',
 'Y': '0.6730000305175782',
 'xG': '0.050078392028808594',
 'player': 'Bruno Fernandes',
 'h_a': 'h',
 'player_id': '1228',
 'situation': 'OpenPlay',
 'season': '2024',
 'shotType': 'LeftFoot',
 'match_id': '26602',
 'h_team': 'Manchester United',
 'a_team': 'Fulham',
 'h_goals': '1',
 'a_goals': '0',
 'date': '2024-08-16 19:00:00',
 'player_assisted': 'Lisandro Martínez',
 'lastAction': 'Chipped'}

In [8]:
# now let's get all of the data for a player, with this method you need to use the player_id
# This is literally every shot they ever took. 

player_data = client.player(player='1228').get_shot_data()
player_data[0]

{'id': '41640',
 'minute': '79',
 'result': 'BlockedShot',
 'X': '0.730999984741211',
 'Y': '0.5059999847412109',
 'xG': '0.031210700049996376',
 'player': 'Bruno Fernandes',
 'h_a': 'h',
 'player_id': '1228',
 'situation': 'OpenPlay',
 'season': '2014',
 'shotType': 'RightFoot',
 'match_id': '5158',
 'h_team': 'Udinese',
 'a_team': 'Empoli',
 'h_goals': '2',
 'a_goals': '0',
 'date': '2014-08-31 19:45:00',
 'player_assisted': 'Cyril Théréau',
 'lastAction': 'Pass'}

In [9]:
df = pd.DataFrame(player_data)

In [10]:
df

,id,minute,result,X,Y,xG,player,h_a,player_id,situation,season,shotType,match_id,h_team,a_team,h_goals,a_goals,date,player_assisted,lastAction
0,41640,79,BlockedShot,0.730999984741211,0.5059999847412109,0.031210700049996376,Bruno Fernandes,h,1228,OpenPlay,2014,RightFoot,5158,Udinese,Empoli,2,0,2014-08-31 19:45:00,Cyril Théréau,Pass
1,41641,82,MissedShots,0.9569999694824218,0.6170000076293946,0.11918000131845474,Bruno Fernandes,h,1228,OpenPlay,2014,RightFoot,5158,Udinese,Empoli,2,0,2014-08-31 19:45:00,Antonio Di Natale,Chipped
2,41643,86,SavedShot,0.77,0.5170000076293946,0.037953298538923264,Bruno Fernandes,h,1228,OpenPlay,2014,RightFoot,5158,Udinese,Empoli,2,0,2014-08-31 19:45:00,Panagiotis Kone,Pass
3,41807,47,BlockedShot,0.759000015258789,0.664000015258789,0.01848829910159111,Bruno Fernandes,a,1228,OpenPlay,2014,RightFoot,5110,Juventus,Udinese,2,0,2014-09-13 19:45:00,Antonio Di Natale,LayOff
4,42170,42,BlockedShot,0.754000015258789,0.534000015258789,0.02486089989542961,Bruno Fernandes,h,1228,OpenPlay,2014,LeftFoot,5125,Udinese,Napoli,1,0,2014-09-21 17:00:00,Emmanuel Badu,Pass
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
660,615460,91,BlockedShot,0.779000015258789,0.23899999618530274,0.010027021169662476,Bruno Fernandes,h,1228,OpenPlay,2024,LeftFoot,26838,Manchester United,Crystal Palace,0,2,2025-02-02 14:00:00,Amad Diallo Traore,Pass
661,617808,24,BlockedShot,0.725999984741211,0.509000015258789,0.017189664766192436,Bruno Fernandes,a,1228,OpenPlay,2024,RightFoot,26850,Tottenham,Manchester United,1,0,2025-02-16 16:30:00,Alejandro Garnacho,Pass
662,618234,70,BlockedShot,0.735,0.6930000305175781,0.03905314952135086,Bruno Fernandes,a,1228,DirectFreekick,2024,RightFoot,26855,Everton,Manchester United,2,2,2025-02-22 12:30:00,None,Standard
663,618235,71,Goal,0.779000015258789,0.6609999847412109,0.0653163269162178,Bruno Fernandes,a,1228,DirectFreekick,2024,RightFoot,26855,Everton,Manchester United,2,2,2025-02-22 12:30:00,None,Standard


In [12]:
df.to_csv('bruno_fernandes_all_shots.csv', index=False)

#### Now let's scrape it ourselves with requests and BeautifulSoup. This is a bit more work so we'll look at only a single match
You'll need to install:
- requests
- BeautifulSoup
- lxml

In [13]:
import re
import requests
import json

from bs4 import BeautifulSoup

response = requests.get("https://understat.com/match/27537")
print(response.status_code)

200


In [14]:
soup = BeautifulSoup(response.content, 'html.parser')
ugly_soup = str(soup)

In [19]:
# strip unnecessary symbols and get only JSON data
shots_data = re.search(r"var shotsData .*= JSON.parse\('(.*)'\)", ugly_soup).group(1)
data = json.loads(shots_data.encode('utf8').decode('unicode_escape'))

In [ ]:
data

{'h': [{'id': '609554',
   'minute': '4',
   'result': 'MissedShots',
   'X': '0.915',
   'Y': '0.515',
   'xG': '0.1274878978729248',
   'player': 'Dusan Vlahovic',
   'h_a': 'h',
   'player_id': '7084',
   'situation': 'OpenPlay',
   'season': '2024',
   'shotType': 'Head',
   'match_id': '27537',
   'h_team': 'Juventus',
   'a_team': 'Fiorentina',
   'h_goals': '2',
   'a_goals': '2',
   'date': '2024-12-29 17:00:00',
   'player_assisted': 'Francisco Conceição',
   'lastAction': 'Cross'},
  {'id': '609556',
   'minute': '19',
   'result': 'Goal',
   'X': '0.860999984741211',
   'Y': '0.585',
   'xG': '0.09523896127939224',
   'player': 'Kephren Thuram',
   'h_a': 'h',
   'player_id': '7265',
   'situation': 'OpenPlay',
   'season': '2024',
   'shotType': 'RightFoot',
   'match_id': '27537',
   'h_team': 'Juventus',
   'a_team': 'Fiorentina',
   'h_goals': '2',
   'a_goals': '2',
   'date': '2024-12-29 17:00:00',
   'player_assisted': 'Manuel Locatelli',
   'lastAction': 'Pass'},
  {

In [25]:
h_df = pd.DataFrame(data['h'])
h_df

,id,minute,result,X,Y,xG,player,h_a,player_id,situation,season,shotType,match_id,h_team,a_team,h_goals,a_goals,date,player_assisted,lastAction
0,609554,4,MissedShots,0.915,0.515,0.1274878978729248,Dusan Vlahovic,h,7084,OpenPlay,2024,Head,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,Francisco Conceição,Cross
1,609556,19,Goal,0.860999984741211,0.585,0.09523896127939224,Kephren Thuram,h,7265,OpenPlay,2024,RightFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,Manuel Locatelli,Pass
2,609562,40,SavedShot,0.9309999847412109,0.6029999923706054,0.4345000982284546,Dusan Vlahovic,h,7084,SetPiece,2024,LeftFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,None,None
3,609563,42,MissedShots,0.9209999847412109,0.5129999923706055,0.047720082104206085,Federico Gatti,h,10993,FromCorner,2024,Head,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,Teun Koopmeiners,Aerial
4,609564,44,MissedShots,0.7780000305175782,0.41700000762939454,0.01350671611726284,Manuel Locatelli,h,1122,OpenPlay,2024,RightFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,None,Aerial
5,609565,47,MissedShots,0.7559999847412109,0.5129999923706055,0.012308514676988125,Dusan Vlahovic,h,7084,OpenPlay,2024,LeftFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,None,Aerial
6,609566,47,Goal,0.915,0.5420000076293945,0.4918035566806793,Kephren Thuram,h,7265,OpenPlay,2024,RightFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,None,None
7,609568,63,BlockedShot,0.909000015258789,0.615,0.06164499744772911,Federico Gatti,h,10993,FromCorner,2024,RightFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,Kenan Yildiz,Pass
8,609569,71,BlockedShot,0.8590000152587891,0.27600000381469725,0.04325911030173302,Francisco Conceição,h,13052,OpenPlay,2024,LeftFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,Manuel Locatelli,Pass
9,609570,73,BlockedShot,0.8640000152587891,0.6659999847412109,0.06881710886955261,Kenan Yildiz,h,11942,OpenPlay,2024,RightFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,Manuel Locatelli,BallTouch


In [26]:
a_df = pd.DataFrame(data['a'])
a_df

,id,minute,result,X,Y,xG,player,h_a,player_id,situation,season,shotType,match_id,h_team,a_team,h_goals,a_goals,date,player_assisted,lastAction
0,609555,18,BlockedShot,0.7469999694824219,0.3029999923706055,0.030985459685325623,Andrea Colpani,a,6718,OpenPlay,2024,LeftFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,Riccardo Sottil,Pass
1,609557,22,MissedShots,0.815,0.49,0.07099736481904984,Moise Kean,a,1304,OpenPlay,2024,LeftFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,Danilo Cataldi,TakeOn
2,609558,27,BlockedShot,0.76,0.675,0.051893025636672974,Riccardo Sottil,a,6672,DirectFreekick,2024,RightFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,None,Standard
3,609559,27,MissedShots,0.8169999694824219,0.5579999923706055,0.03693047910928726,Yacine Adli,a,6762,SetPiece,2024,LeftFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,None,Rebound
4,609560,30,BlockedShot,0.81,0.524000015258789,0.05092712119221687,Albert Gudmundsson,a,10366,OpenPlay,2024,RightFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,Dodô,Pass
5,609561,37,Goal,0.9540000152587891,0.46799999237060547,0.49202555418014526,Moise Kean,a,1304,OpenPlay,2024,Head,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,Yacine Adli,Cross
6,609567,60,SavedShot,0.8769999694824219,0.6509999847412109,0.09770980477333069,Moise Kean,a,1304,OpenPlay,2024,RightFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,Lucas Beltrán,BallRecovery
7,609573,75,SavedShot,0.78,0.3029999923706055,0.013003279455006123,Dodô,a,10935,OpenPlay,2024,RightFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,Jonathan Ikone,TakeOn
8,609575,82,BlockedShot,0.8030000305175782,0.40700000762939453,0.04392712190747261,Lucas Beltrán,a,11900,OpenPlay,2024,LeftFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,Amir Richardson,LayOff
9,609577,86,MissedShots,0.8969999694824219,0.515,0.0886312872171402,Moise Kean,a,1304,OpenPlay,2024,LeftFoot,27537,Juventus,Fiorentina,2,2,2024-12-29 17:00:00,None,Dispossessed
